# Chapter 04 — Regression Models: Live Examples

**Session 2 | Chapter 1 | Duration: ~10 min**

We compare four regression models on the **California Housing dataset** (predict median house price).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Ready!')

## 1. Load and Explore the Data

In [ ]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target  # target: median house value (in $100k)

print('Shape:', df.shape)
print('Target: median house value (in $100,000)')
df.head()

In [ ]:
print(df.describe().round(2))

In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 2. Define a Scoring Helper

We create a helper function to evaluate every model the same way.

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    print(f'{name:<25}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}')
    return y_pred

print(f'{"Model":<25}  {"MAE":>10}  {"RMSE":>10}  {"R²":>8}')
print('-' * 58)

## 3. Linear Regression

In [ ]:
# Linear regression needs scaling for fair coefficient comparison
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
preds_lr = evaluate_model('Linear Regression', lr_pipe, X_train, X_test, y_train, y_test)

# Show coefficients (what each feature contributes)
coef = pd.Series(
    lr_pipe.named_steps['model'].coef_,
    index=housing.feature_names
).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
coef.plot(kind='barh', ax=ax, color=['#e74c3c' if v < 0 else '#2ecc71' for v in coef])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Linear Regression Coefficients (after standardization)', fontsize=12)
ax.set_xlabel('Coefficient value')
plt.tight_layout()
plt.show()

## 4. Ridge Regression (L2 Regularization)

In [ ]:
ridge_pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
preds_ridge = evaluate_model('Ridge (alpha=1.0)', ridge_pipe, X_train, X_test, y_train, y_test)

# Compare Ridge coefficients across different alpha values
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
coef_df = pd.DataFrame(index=housing.feature_names)

for alpha in alphas:
    p = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha))])
    p.fit(X_train, y_train)
    coef_df[f'α={alpha}'] = p.named_steps['model'].coef_

print('\nRidge coefficients vs alpha (how regularization shrinks them):')
print(coef_df.round(3))

## 5. Lasso Regression (L1 — Feature Selection)

In [ ]:
lasso_pipe = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.01))])
preds_lasso = evaluate_model('Lasso (alpha=0.01)', lasso_pipe, X_train, X_test, y_train, y_test)

lasso_coef = pd.Series(
    lasso_pipe.named_steps['model'].coef_,
    index=housing.feature_names
)
print('\nLasso coefficients (zeros = features eliminated):')
print(lasso_coef.round(4))
zero_features = lasso_coef[lasso_coef == 0].index.tolist()
print(f'\nFeatures set to zero by Lasso: {zero_features}')

## 6. Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
preds_rf = evaluate_model('Random Forest', rf, X_train, X_test, y_train, y_test)

# Feature importances
importances = pd.Series(rf.feature_importances_, index=housing.feature_names).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='#3498db', alpha=0.8)
ax.set_title('Random Forest Feature Importances', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()
print('\nMedInc (median income) is by far the most important feature!')

## 7. Visual Comparison: Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models_preds = [
    ('Linear Regression', preds_lr, '#3498db'),
    ('Random Forest', preds_rf, '#2ecc71'),
]

for ax, (name, preds, color) in zip(axes, models_preds):
    ax.scatter(y_test, preds, alpha=0.3, s=10, color=color)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
            'r--', linewidth=2, label='Perfect prediction')
    r2 = r2_score(y_test, preds)
    ax.set_title(f'{name} (R²={r2:.3f})', fontsize=12)
    ax.set_xlabel('Actual House Value ($100k)')
    ax.set_ylabel('Predicted House Value ($100k)')
    ax.legend()

plt.suptitle('Predicted vs Actual (closer to red line = better)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

| Model | Best for | Key insight |
|-------|---------|------------|
| Linear Regression | Interpretability | Coefficients tell a story |
| Ridge | Correlated features | Shrinks without eliminating |
| Lasso | Feature selection | Zeroes out irrelevant features |
| Random Forest | Best accuracy | Non-linear, feature importances |

**Random Forest wins on accuracy, but Linear Regression wins on interpretability.**  
Always consider the trade-off based on your use case!